In [ ]:
!pip install pyDOE

In [ ]:
import tensorflow as tf
import numpy as np
from pyDOE import lhs
import matplotlib.pyplot as plt
import math

from scipy.integrate import odeint

In [ ]:
# Boundaries of the computational domain
t0, tfinal = 0.0, 5.0

### Non-invariant version

In [ ]:
# Initial conditions
x0 = 1.0
y0 = 1.0

xt0 = 0.5
yt0 = 0.0

init_data = (x0, y0, xt0, yt0)

In [ ]:
H0 = (1/2)*(xt0**2 + yt0**2) - (1/math.sqrt(x0**2 + y0**2))
print(H0)

A0 = x0*yt0 - y0*xt0
print(A0)

In [ ]:
# Define the network
def build_model(nr_units = 60, nr_layers=4, summary=False):

  inp = b = tf.keras.layers.Input(shape=(1,))

  for i in range(nr_layers):
    b = tf.keras.layers.Dense(nr_units, activation='tanh')(b)
  outx = tf.keras.layers.Dense(1, activation='linear')(b)
  outy = tf.keras.layers.Dense(1, activation='linear')(b)

  #outx = tf.keras.layers.Lambda(lambda x: x0 + x[0]*xt0 + (x[0]/tfinal)**2*x[1])([inp, outx])
  #outy = tf.keras.layers.Lambda(lambda x: y0 + x[0]*yt0 + (x[0]/tfinal)**2*x[1])([inp, outy])

  model = tf.keras.models.Model(inp, [outx, outy])

  if summary:
    model.summary()

  return model

In [ ]:
model = build_model(summary = True)

In [ ]:
# Define the collocation points over the domain [t_0, t_1]
def defineCollocationPoints(t_bdry, N_pde=100):

    # Sample points where to evaluate the PDE
  ode_points = t_bdry[0] + (t_bdry[1] - t_bdry[0])*lhs(1, N_pde)

  return ode_points

In [ ]:
# Define the collocation points
de_points = defineCollocationPoints([t0, tfinal], 500)

In [ ]:
plt.plot(de_points[:,0], 0*de_points[:,0],'.')
plt.xlabel('t')
plt.title('Collocation points being used')
plt.savefig('CollocationPoints1D.png')

In [ ]:
alpha = 1
beta = 1
gamma = 1

In [ ]:
# The main training function
@tf.function
def train_network(odes, model):

  # DE collocation points
  t_de = odes[:,:1]

  # Outer gradient for tuning network parameters
  with tf.GradientTape() as tape:

    # Inner gradient for derivatives of u wrt x and t
    with tf.GradientTape(persistent=True) as tape2:
      tape2.watch(t_de)
      with tf.GradientTape(persistent=True) as tape3:
        tape3.watch(t_de)
        [x, y] = model(t_de)

        x = x0 + t_de*xt0 + (1/2)*(t_de**2)*x
        y = y0 + t_de*yt0 + (1/2)*(t_de**2)*y

      xt = tape3.gradient(x, t_de)
      yt = tape3.gradient(y, t_de)
    xtt = tape2.gradient(xt, t_de)
    ytt = tape2.gradient(yt, t_de)

    # Define the differential equation loss
    eqn1 = xtt + x/tf.sqrt((x**2+y**2)**3)
    eqn2 = ytt + y/tf.sqrt((x**2+y**2)**3)

    H = (1/2)*(xt**2 + yt**2) - (1/tf.math.sqrt(x**2 + y**2))
    A = x*yt - y*xt


    deloss = tf.reduce_mean(tf.square(eqn1) + tf.square(eqn2))
    hloss = tf.square(tf.reduce_mean(H) - H0)
    aloss = tf.square(tf.reduce_mean(A) - A0)

    loss = alpha*deloss + beta*hloss + gamma*aloss

  grads = tape.gradient(loss, model.trainable_variables)
  return deloss, hloss, aloss, grads

In [ ]:
def PINNtrain(de_points, epochs=1000):

  # Total number of collocation points
  N_de = len(de_points)

  # Batch size
  bs_de = N_de

  # Learning rate
  lr_model = 1e-3

  epoch_loss = np.zeros(epochs)
  nr_batches = 0

  # Generate the tf.Dataset for the differential equations points
  ds = tf.data.Dataset.from_tensor_slices(de_points.astype(np.float32))
  ds = ds.cache().shuffle(N_de).batch(bs_de)

  # Generate the model
  opt = tf.keras.optimizers.Adam(lr_model)
  model = build_model()

  # Main training loop
  for i in range(epochs):

    # Training for that epoch
    for des in ds:

      # Train the network
      DEloss, Hloss, Aloss, grads = train_network(des, model)

      # Gradient step
      opt.apply_gradients(zip(grads, model.trainable_variables))

      epoch_loss[i] += alpha*DEloss + beta*Hloss + gamma*Aloss
      nr_batches += 1

    # Get total epoch loss
    epoch_loss[i] /= nr_batches
    nr_batches = 0

    if (np.mod(i, 100)==0):
      print("DE loss, H loss in {}th epoch: {: 6.4f}, {: 6.4f}, {: 6.4f}.".format(i, DEloss, Hloss, Aloss))

  return epoch_loss, model

In [ ]:
epochs = 5000
loss, model = PINNtrain(de_points, epochs)

In [ ]:
# Initial angular momentum
H0 = y0 * xt0 - x0 * yt0

# Grid where to evaluate the model
m = 200
t = np.linspace(t0, tfinal, m)

# Model prediction
tt = tf.cast(tf.Variable(np.expand_dims(t, axis=1)), tf.float32)
with tf.GradientTape(persistent=True) as tape:
  tape.watch(tt)
  [x, y] = model(tt)

  x = x0 + tt*xt0 + (1/2)*(tt**2)*x
  y = y0 + tt*yt0 + (1/2)*(tt**2)*y

xt = tape.gradient(x, tt)
yt = tape.gradient(y, tt)

H = y*xt - x*yt

In [ ]:
plt.plot(t, H-H0)
plt.xlabel('t')
plt.ylabel('Conservation Error H')

In [ ]:
## RK reference solution
def kepler(w, t):
  x, y, xt, yt = w
  dwdt = [xt, yt, -x/np.sqrt((x**2+y**2)**3), -y/np.sqrt((x**2+y**2)**3)]
  return dwdt

urk = odeint(kepler, init_data, t)

In [ ]:
plt.plot(t, x[:,0],'b')
plt.plot(t, urk[:,0],'b--')

plt.plot(t, y[:,0],'r')
plt.plot(t, urk[:,1],'r--')
plt.legend(['NN solution', 'RK reference solution'])

In [ ]:
plt.plot(x[:,0], y[:,0])
plt.plot(urk[:,0], urk[:,1])
plt.xlabel('x')
plt.ylabel('y')
plt.grid()

plt.legend(['NN','RK'])

In [ ]:
# Errors
print(f'MSE: {np.mean((x[:,0]-urk[:,0])**2)}')
print(f'MAE: {np.mean(np.abs(x[:,0]-urk[:,0]))}')

In [ ]:
# Plot the solution
fig = plt.figure(figsize=(21,7))

# Plot the numerical and exact solutions
plt.subplot(131)
plt.plot(t, x[:,0])
plt.plot(t, urk[:,0],'k--')
plt.grid()
plt.xlabel('t')
plt.ylabel('u')
plt.legend(['NN solution','RK reference solution'])

# Plot the error
plt.subplot(132)
plt.plot(t, x[:,0]-urk[:,0])
plt.grid()
plt.xlabel('t')
plt.ylabel('error')

# Plot the loss function
plt.subplot(133)
plt.semilogy(np.linspace(1, epochs, epochs),loss)
plt.grid()
plt.xlabel('epochs')
plt.ylabel('loss')

plt.savefig('PINNODE.png')